```text
是同一个数学概念，但在 Python 生态里，BrainFlow 里的小波更接近 PyWavelets，而不是 SciPy 的旧小波接口。

结论

BrainFlow 里的：

DataFilter.perform_wavelet_denoising(...)

使用的是小波变换 + 阈值去噪思路。

它和 PyWavelets 里的：

pywt.wavedec(...)
pywt.threshold(...)
pywt.waverec(...)

属于同一类东西：离散小波变换 DWT / 多层小波分解 / 小波阈值去噪。

而 SciPy 里的小波功能主要是 scipy.signal.cwt、morlet、ricker 这类连续小波变换 CWT相关接口，并且 SciPy 官方已经把 signal.cwt、signal.morlet 等接口标记为 deprecated，建议改用 PyWavelets。

⸻

三者关系可以这样理解

库	主要用途	和 BrainFlow 的关系
BrainFlow	给 EEG/生理信号提供现成去噪接口	封装好了小波去噪流程
PyWavelets / pywt	Python 中专门做小波分析的库	概念最接近 BrainFlow
SciPy signal.cwt	连续小波变换，常用于时频分析	和 BrainFlow 去噪不是同一个常用接口方向

BrainFlow 官方示例中 perform_wavelet_denoising 的调用包含小波类型、分解层数、去噪方法、阈值类型、边界扩展方式、噪声估计层级等参数，说明它做的是完整的“小波阈值去噪”流程。

⸻

举个对应关系

BrainFlow 写法：

from brainflow.data_filter import DataFilter, WaveletTypes
denoised = noisy_signal.copy()
DataFilter.perform_wavelet_denoising(
    data=denoised,
    wavelet=WaveletTypes.DB4,
    decomposition_level=3
)

在 PyWavelets 里大致对应这种思路：

import pywt
import numpy as np
coeffs = pywt.wavedec(noisy_signal, wavelet='db4', level=3)
# 对细节系数做阈值处理
threshold = np.std(coeffs[-1]) * np.sqrt(2 * np.log(len(noisy_signal)))
coeffs_thresh = [coeffs[0]] + [
    pywt.threshold(c, threshold, mode='soft')
    for c in coeffs[1:]
]
denoised = pywt.waverec(coeffs_thresh, wavelet='db4')

不是说 BrainFlow 内部一定逐行等价于这段代码，而是算法结构类似：

原始信号
  ↓
多层小波分解
  ↓
对高频细节系数做阈值处理
  ↓
小波重构
  ↓
去噪信号

PyWavelets 本身就是专门的 Python 小波变换库，支持 DWT、IDWT、SWT、CWT、Wavelet Packet 等多种小波变换。

⸻

关键区别

1. BrainFlow 是“封装好的去噪”

你调用：

DataFilter.perform_wavelet_denoising(...)

它直接修改 data，最后得到去噪后的信号。

你不需要自己处理：

coeffs = ...
threshold = ...
waverec = ...

所以 BrainFlow 更方便，但自由度略低。

⸻

2. PyWavelets 更灵活

用 pywt 时，你可以精细控制：

wavelet='db4'
level=3
mode='symmetric'
threshold=...
threshold mode='soft' / 'hard'

适合做实验、调参、论文复现。

⸻

3. SciPy 的小波不推荐作为主要选择

SciPy 以前有：

scipy.signal.cwt
scipy.signal.ricker
scipy.signal.morlet

但这些更偏向连续小波变换 CWT，常用于时频分析，不是 EEG 去噪里最常用的多层 DWT 阈值去噪流程。而且 SciPy 文档已说明 signal.cwt 在 SciPy 1.12 被弃用，并建议使用 PyWavelets。

⸻

所以你可以这样记

BrainFlow 小波去噪：

= 工程封装版小波阈值去噪
≈ PyWavelets 的 wavedec + threshold + waverec
≠ SciPy signal.cwt 的主要使用场景

如果你只是做 EEG 去噪，用 BrainFlow 足够方便。

如果你要深入研究不同小波、阈值策略、每层系数变化，建议用 PyWavelets。

如果你要做时频图，例如看某个频率随时间怎么变化，才更多考虑 CWT，也就是 PyWavelets 的 pywt.cwt。
```